![Banner](banner.jpg)

# Laboratorio 9: Transferencia de Aprendizaje (Transfer Learning)

## 1. Introducción

### ¿Qué es Transfer Learning?

La **transferencia de aprendizaje** es una técnica que permite reutilizar un modelo
preentrenado en una tarea (generalmente en un dataset muy grande) como punto de partida
para una tarea diferente pero relacionada. En lugar de entrenar una red desde cero,
aprovechamos los patrones ya aprendidos.

Las capas iniciales de una red convolucional aprenden características genéricas
(bordes, texturas, colores) que son útiles para cualquier tarea de visión.
Las capas más profundas aprenden patrones más específicos de la tarea original.
Al reutilizar estas capas, podemos lograr buenos resultados incluso con pocos datos.

### Dos estrategias principales

1. **Feature Extraction**: Congelamos el modelo preentrenado y solo entrenamos un nuevo
   clasificador encima. El modelo base actúa como un extractor de características fijo.

2. **Fine-Tuning**: Además de entrenar el clasificador, descongelamos algunas capas
   superiores del modelo base y las reentrenamos con un learning rate bajo. Esto permite
   adaptar las características aprendidas a nuestra tarea específica.

## 2. Preparación del entorno

In [ ]:
!pip install tensorflow matplotlib numpy kagglehub scikit-learn

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
import shutil
import random
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32

## 3. Carga del dataset

Utilizaremos el dataset **Wheat Plant Diseases** que contiene imágenes de hojas de trigo
clasificadas en diferentes categorías de enfermedad.

In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download('kushagra3204/wheat-plant-diseases')
print(f'Dataset descargado en: {dataset_path}')

In [ ]:
# Explorar la estructura del dataset
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    folder_name = os.path.basename(root)
    img_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    if img_files:
        print(f'{indent}{folder_name}/ ({len(img_files)} imágenes)')
    elif level < 3:
        print(f'{indent}{folder_name}/')

In [ ]:
# Encontrar el directorio de datos
data_dir = None
for p in Path(dataset_path).rglob('*'):
    if p.is_dir():
        subdirs = [d for d in p.iterdir() if d.is_dir()]
        if len(subdirs) >= 2:
            has_imgs = any(
                f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']
                for d in subdirs for f in d.iterdir() if f.is_file()
            )
            if has_imgs:
                data_dir = p
                break

if data_dir is None:
    data_dir = Path(dataset_path)

print(f'Directorio de datos: {data_dir}')
classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
print(f'Clases encontradas ({len(classes)}): {classes}')

# Contar imágenes por clase
for cls in classes:
    cls_dir = data_dir / cls
    count = len([f for f in cls_dir.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']])
    print(f'  {cls}: {count} imágenes')

## 4. Exploración de datos

In [ ]:
# Visualizar muestras de cada clase
n_classes = len(classes)
cols = min(n_classes, 5)
rows = (n_classes + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)

for idx, cls in enumerate(classes):
    row, col = idx // cols, idx % cols
    cls_dir = data_dir / cls
    img_files = sorted([f for f in cls_dir.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']])
    if img_files:
        img = tf.keras.utils.load_img(img_files[0], target_size=(IMG_SIZE, IMG_SIZE))
        axes[row, col].imshow(img)
    axes[row, col].set_title(cls, fontsize=11)
    axes[row, col].axis('off')

# Ocultar ejes vacíos
for idx in range(len(classes), rows * cols):
    row, col = idx // cols, idx % cols
    axes[row, col].axis('off')

plt.suptitle('Muestras del dataset Wheat Plant Diseases', fontsize=14)
plt.tight_layout()
plt.savefig('wheat_samples.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Distribución de clases
class_counts = {}
for cls in classes:
    cls_dir = data_dir / cls
    class_counts[cls] = len([f for f in cls_dir.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(class_counts.keys(), class_counts.values(), color='forestgreen')
ax.set_xlabel('Clase')
ax.set_ylabel('Número de imágenes')
ax.set_title('Distribución de imágenes por clase')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('wheat_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Preparación de datos

In [ ]:
# Crear datasets de TensorFlow
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Dividir validación y test
val_size = int(0.5 * len(val_test_ds))
val_ds = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f'Clases: {class_names}')
print(f'Número de clases: {num_classes}')

In [ ]:
# Normalizar con preprocess_input de MobileNetV2
# MobileNetV2 espera valores en rango [-1, 1]
normalization_layer = layers.Rescaling(1./127.5, offset=-1)

train_ds_norm = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds_norm = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds_norm = test_ds.map(lambda x, y: (normalization_layer(x), y))

# Optimizar rendimiento con prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds_norm = train_ds_norm.prefetch(buffer_size=AUTOTUNE)
val_ds_norm = val_ds_norm.prefetch(buffer_size=AUTOTUNE)
test_ds_norm = test_ds_norm.prefetch(buffer_size=AUTOTUNE)

## 6. Modelo base preentrenado

Vamos a usar **MobileNetV2**, un modelo ligero y eficiente preentrenado en **ImageNet**
(1.4M de imágenes en 1000 clases). Cargamos el modelo **sin la capa de clasificación**
(`include_top=False`) para poder agregar nuestro propio clasificador.

In [ ]:
# Cargar MobileNetV2 preentrenado
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

print(f'Número de capas en MobileNetV2: {len(base_model.layers)}')
print(f'Forma de la salida del modelo base: {base_model.output_shape}')

In [ ]:
# Visualizar qué tipo de features extrae el modelo base
# Tomamos una imagen y vemos las activaciones de capas intermedias
sample_batch = next(iter(train_ds_norm))
sample_img = sample_batch[0][0:1]

# Modelo parcial para ver features de una capa intermedia
feature_model = keras.Model(
    inputs=base_model.input,
    outputs=base_model.get_layer(index=10).output
)
features = feature_model.predict(sample_img, verbose=0)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(8):
    row, col = i // 4, i % 4
    axes[row, col].imshow(features[0, :, :, i], cmap='viridis')
    axes[row, col].set_title(f'Filtro {i}')
    axes[row, col].axis('off')
plt.suptitle('Feature maps extraídos por MobileNetV2 (capa intermedia)', fontsize=14)
plt.tight_layout()
plt.savefig('feature_maps.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Feature Extraction

En esta estrategia:
1. **Congelamos** todas las capas del modelo base (no se actualizan durante el entrenamiento)
2. Agregamos nuestro propio clasificador encima
3. Solo entrenamos las capas nuevas

Esto es rápido y funciona bien cuando el nuevo dataset es similar al original.

In [ ]:
# Congelar el modelo base
base_model.trainable = False

# Construir modelo con feature extraction
fe_model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
], name='feature_extraction_model')

fe_model.summary()

In [ ]:
# Compilar y entrenar
fe_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Entrenando con Feature Extraction...')
fe_history = fe_model.fit(
    train_ds_norm,
    validation_data=val_ds_norm,
    epochs=15,
    verbose=1
)

In [ ]:
# Evaluar en test
fe_test_loss, fe_test_acc = fe_model.evaluate(test_ds_norm, verbose=0)
print(f'\nFeature Extraction - Test accuracy: {fe_test_acc:.4f}')
print(f'Feature Extraction - Test loss: {fe_test_loss:.4f}')

In [ ]:
# Graficar curvas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(fe_history.history['loss'], label='Entrenamiento', linewidth=2)
ax1.plot(fe_history.history['val_loss'], label='Validación', linewidth=2)
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.set_title('Feature Extraction - Pérdida')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(fe_history.history['accuracy'], label='Entrenamiento', linewidth=2)
ax2.plot(fe_history.history['val_accuracy'], label='Validación', linewidth=2)
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.set_title('Feature Extraction - Precisión')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Curvas de aprendizaje — Feature Extraction', fontsize=14)
plt.tight_layout()
plt.savefig('fe_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Fine-Tuning

Ahora vamos a **descongelar** las últimas capas del modelo base y reentrenarlas
con un **learning rate muy bajo**. Esto permite adaptar las features del modelo
a nuestro dataset específico. Es importante usar un learning rate bajo (1e-5) para no destruir los pesos preentrenados.

In [ ]:
# Descongelar las últimas 30 capas del modelo base
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Contar parámetros entrenables
trainable_count = sum(p.numpy().size for p in fe_model.trainable_weights)
total_count = sum(p.numpy().size for p in fe_model.weights)
print(f'Parámetros totales: {total_count:,}')
print(f'Parámetros entrenables (fine-tuning): {trainable_count:,}')
print(f'Parámetros congelados: {total_count - trainable_count:,}')

In [ ]:
# Recompilar con learning rate bajo
fe_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('\nEntrenando con Fine-Tuning...')
ft_history = fe_model.fit(
    train_ds_norm,
    validation_data=val_ds_norm,
    epochs=10,
    verbose=1
)

In [ ]:
# Evaluar fine-tuning en test
ft_test_loss, ft_test_acc = fe_model.evaluate(test_ds_norm, verbose=0)
print(f'\nFine-Tuning - Test accuracy: {ft_test_acc:.4f}')
print(f'Fine-Tuning - Test loss: {ft_test_loss:.4f}')

In [ ]:
# Graficar curvas de fine-tuning
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ft_history.history['loss'], label='Entrenamiento', linewidth=2)
ax1.plot(ft_history.history['val_loss'], label='Validación', linewidth=2)
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.set_title('Fine-Tuning - Pérdida')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(ft_history.history['accuracy'], label='Entrenamiento', linewidth=2)
ax2.plot(ft_history.history['val_accuracy'], label='Validación', linewidth=2)
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.set_title('Fine-Tuning - Precisión')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Curvas de aprendizaje — Fine-Tuning', fontsize=14)
plt.tight_layout()
plt.savefig('ft_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Comparación: Feature Extraction vs Fine-Tuning

In [ ]:
# Obtener predicciones para reporte detallado
y_true = []
y_pred_ft = []

for images, labels in test_ds_norm:
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    predictions = fe_model.predict(images, verbose=0)
    y_pred_ft.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred_ft = np.array(y_pred_ft)

print('=== Reporte de clasificación (Fine-Tuning) ===')
print(classification_report(y_true, y_pred_ft, target_names=class_names))

In [ ]:
# Matriz de confusión
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred_ft)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.xticks(rotation=45, ha='right')
plt.title('Matriz de confusión — Fine-Tuning', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_ft.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Comparación de resultados
print('\n' + '='*50)
print('COMPARACIÓN DE RESULTADOS')
print('='*50)
print(f'Feature Extraction - Test Accuracy: {fe_test_acc:.4f}')
print(f'Fine-Tuning        - Test Accuracy: {ft_test_acc:.4f}')
print(f'Mejora con Fine-Tuning: {(ft_test_acc - fe_test_acc)*100:.2f} puntos porcentuales')
print('='*50)

In [ ]:
# Gráfica comparativa
fig, ax = plt.subplots(figsize=(8, 5))
methods = ['Feature Extraction', 'Fine-Tuning']
accuracies = [fe_test_acc, ft_test_acc]
colors = ['#3498db', '#2ecc71']
bars = ax.bar(methods, accuracies, color=colors, width=0.5)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_title('Comparación: Feature Extraction vs Fine-Tuning')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('comparison.png', dpi=100, bbox_inches='tight')
plt.show()